# प्रायोरिटी मेंबर मिडलवेयर के साथ होटल बुकिंग

यह नोटबुक Microsoft एजेंट फ्रेमवर्क का उपयोग करके **फंक्शन-आधारित मिडलवेयर** प्रदर्शित करता है। हम कंडीशनल वर्कफ़्लो उदाहरण पर एक मिडलवेयर लेयर जोड़ते हैं जो प्रायोरिटी मेंबर्स को विशेष सुविधाएं देता है।

## आप क्या सीखेंगे:
1. **फंक्शन-आधारित मिडलवेयर**: फंक्शन परिणामों को इंटरसेप्ट और संशोधित करें
2. **कॉन्टेक्स्ट एक्सेस**: निष्पादन के बाद `context.result` पढ़ें और संशोधित करें
3. **व्यावसायिक लॉजिक कार्यान्वयन**: प्रायोरिटी मेंबर लाभ
4. **परिणाम ओवरराइड**: उपयोगकर्ता की स्थिति के आधार पर फंक्शन परिणाम बदलें
5. **एक जैसा वर्कफ़्लो, अलग परिणाम**: मिडलवेयर संचालित व्यवहार परिवर्तन

## मिडलवेयर के साथ वर्कफ़्लो आर्किटेक्चर:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## कंडीशनल वर्कफ़्लो से मुख्य अंतर:

**मिडलवेयर के बिना** (14-conditional-workflow.ipynb):
- पेरिस में कोई कमरे नहीं → alternative_agent की ओर मार्गदर्शन

**मिडलवेयर के साथ** (यह नोटबुक):
- सामान्य उपयोगकर्ता + पेरिस → कोई कमरे नहीं → alternative_agent की ओर मार्गदर्शन
- प्रायोरिटी उपयोगकर्ता + पेरिस → 🌟 मिडलवेयर ओवरराइड करता है! → उपलब्ध → booking_agent की ओर मार्गदर्शन

## आवश्यकताएँ:
- Microsoft एजेंट फ्रेमवर्क इंस्टॉल किया हुआ हो
- कंडीशनल वर्कफ़्लो की समझ (देखें 14-conditional-workflow.ipynb)
- GitHub टोकन या OpenAI API कुंजी
- मिडलवेयर पैटर्न की बुनियादी समझ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## चरण 1: संरचित आउटपुट के लिए Pydantic मॉडल परिभाषित करें

ये मॉडल **स्कीमा** को परिभाषित करते हैं जिसे एजेंट वापस करेंगे। हमने जोड़ा है `priority_override` फील्ड यह ट्रैक करने के लिए कि जब मिडलवेयर उपलब्धता परिणाम को संशोधित करता है।


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## चरण 2: प्राथमिकता सदस्यों का डेटाबेस परिभाषित करें

इस डेमो के लिए, हम प्राथमिकता सदस्यता डेटाबेस का अनुकरण करेंगे। उत्पादन में, यह एक वास्तविक डेटाबेस या API को क्वेरी करेगा।

**प्राथमिकता सदस्य:**
- `alice@example.com` - वीआईपी सदस्य
- `bob@example.com` - प्रीमियम सदस्य  
- `priority_user` - परीक्षण खाता


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## चरण 3: होटल बुकिंग टूल बनाएं

सशर्त वर्कफ़्लो की तरह ही, लेकिन अब इसे मिडलवेयर द्वारा रोका जाएगा!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## चरण 4: 🌟 प्राथमिकता जांच मिडलवेयर बनाएं (मुख्य फीचर!)

यह इस नोटबुक की **मुख्य कार्यक्षमता** है। मिडलवेयर:

1. होटल_बुकिंग फ़ंक्शन कॉल को **अवरोधित** करता है
2. `next(context)` कॉल करके सामान्य रूप से फ़ंक्शन **चलाता** है
3. `context.result` में परिणाम **निरीक्षण** करता है
4. यदि उपयोगकर्ता प्राथमिकता है और कोई कमरे उपलब्ध नहीं हैं तो परिणाम को **अधिलेखित** करता है
5. संशोधित परिणाम को एजेंट को **वापस करता** है

**मुख्य पैटर्न:**
```python
async def my_middleware(context, next):
    await next(context)  # फ़ंक्शन निष्पादित करें
    # अब context.result में फ़ंक्शन का आउटपुट है
    if some_condition:
        context.result = new_value  # ओवरराइड करें!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## चरण 5: रूटिंग के लिए स्थिति कार्यों को परिभाषित करें

समान स्थिति कार्य जो शर्तीय वर्कफ़्लो के रूप में हैं - वे रूटिंग निर्धारित करने के लिए संरचित आउटपुट का निरीक्षण करते हैं।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## चरण 6: कस्टम डिस्प्ले एक्जीक्यूटर बनाएँ

पहले जैसा ही एक्जीक्यूटर - अंतिम वर्कफ़्लो आउटपुट दिखाता है।


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## कदम 7: पर्यावरण चर लोड करना

LLM क्लाइंट (Microsoft Foundry / Azure OpenAI) को कॉन्फ़िगर करें।


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## चरण 8: मिडलवेयर के साथ AI एजेंट बनाना

**प्रमुख अंतर:** जब हम availability_agent बना रहे होते हैं, तो हम `middleware` पैरामीटर पास करते हैं!

इस तरह हम एजेंट के फंक्शन इनवोकेशन पाइपलाइन में priority_check_middleware इंजेक्ट करते हैं।


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## चरण 9: वर्कफ़्लो बनाएं

पहले की तरह ही वर्कफ़्लो संरचना - उपलब्धता के आधार पर सशर्त मार्गदर्शन।


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## चरण 10: परीक्षण मामला 1 - पेरिस में नियमित उपयोगकर्ता (कोई अधिलेखन नहीं)

एक नियमित उपयोगकर्ता पेरिस बुक करने की कोशिश करता है → कोई कमरे नहीं → alternative_agent पर रूट करता है


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## चरण 11: टेस्ट केस 2 - 🌟 पेरिस में प्रायरिटी यूज़र (ओवरराइड के साथ!)

एक प्रायरिटी सदस्य पेरिस बुक करने की कोशिश करता है → शुरुआत में कोई कमरे नहीं → 🌟 मिडलवेयर ओवरराइड करता है! → बुकिंग_एजेंट को रूट करता है

**यह मिडलवेयर की शक्ति का प्रमुख प्रदर्शन है!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## चरण 12: टेस्ट केस 3 - स्टॉकहोम में प्राथमिकता उपयोगकर्ता (पहले से उपलब्ध)

प्राथमिकता उपयोगकर्ता स्टॉकहोम का प्रयास करता है → कमरे उपलब्ध हैं → कोई ओवरराइड आवश्यक नहीं → booking_agent को रूट करता है

यह दिखाता है कि मिडलवेयर केवल तभी काम करता है जब आवश्यकता हो!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## प्रमुख निष्कर्ष और मिडलवेयर अवधारणाएँ

### ✅ आपने क्या सीखा:

#### **1. फ़ंक्शन-आधारित मिडलवेयर पैटर्न**

मिडलवेयर एक सरल async फ़ंक्शन का उपयोग करके फ़ंक्शन कॉल्स को इंटरसेप्ट करता है:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # फ़ंक्शन निष्पादन से पहले
    print("Intercepting...")
    
    # फ़ंक्शन चलाएं
    await next(context)
    
    # फ़ंक्शन निष्पादन के बाद - परिणाम जांचें
    if context.result:
        # आवश्यकता होने पर परिणाम संशोधित करें
        context.result = modified_value
```

#### **2. संदर्भ पहुंच और परिणाम अधिलेखित करना**

- `context.function` - कॉल किए जा रहे फ़ंक्शन तक पहुंच
- `context.arguments` - फ़ंक्शन तर्क पढ़ें
- `context.kwargs` - अतिरिक्त पैरामीटर तक पहुंच
- `await next(context)` - फ़ंक्शन निष्पादित करें
- `context.result` - फ़ंक्शन के आउटपुट को पढ़ें/संशोधित करें

#### **3. व्यावसायिक तर्क कार्यान्वयन**

हमारा मिडलवेयर प्राथमिकता सदस्य लाभों को लागू करता है:
- **नियमित उपयोगकर्ता**: कोई संशोधन नहीं, मानक कार्यप्रवाह
- **प्राथमिकता उपयोगकर्ता**: "कोई उपलब्धता नहीं" → "उपलब्ध" अधिलेखित करें
- **सशर्त तर्क**: केवल आवश्यक होने पर अधिलेखित करता है

#### **4. एक ही कार्यप्रवाह, विभिन्न परिणाम**

मिडलवेयर की शक्ति:
- ✅ कार्यप्रवाह संरचना में कोई बदलाव नहीं
- ✅ टूल फ़ंक्शन में कोई बदलाव नहीं
- ✅ सशर्त रूटिंग तर्क में कोई बदलाव नहीं
- ✅ केवल मिडलवेयर → अलग व्यवहार!

### 🚀 वास्तविक दुनिया के अनुप्रयोग:

1. **VIP/प्रिमियम फीचर्स**
   - प्रिमियम उपयोगकर्ताओं के लिए रेट लिमिट्स अधिलेखित करें
   - संसाधनों के लिए प्राथमिकता पहुंच प्रदान करें
   - प्रिमियम फीचर्स को गतिशील रूप से अनलॉक करें

2. **A/B परीक्षण**
   - उपयोगकर्ताओं को विभिन्न कार्यान्वयनों पर रूट करें
   - विशिष्ट उपयोगकर्ताओं के साथ नई विशेषताओं का परीक्षण करें
   - सुविधाओं का क्रमिक रोलआउट

3. **सुरक्षा और अनुपालन**
   - फ़ंक्शन कॉल्स का ऑडिट करें
   - संवेदनशील ऑपरेशनों को ब्लॉक करें
   - व्यावसायिक नियम लागू करें

4. **प्रदर्शन अनुकूलन**
   - विशिष्ट उपयोगकर्ताओं के लिए परिणाम कैश करें
   - संभव होने पर महंगे ऑपरेशन छोड़ दें
   - गतिशील संसाधन आवंटन

5. **त्रुटि प्रबंधन और पुनः प्रयास**
   - त्रुटियों को पकड़ें और सावधानीपूर्वक संभालें
   - पुनः प्रयास तर्क लागू करें
   - वैकल्पिक कार्यान्वयन पर फालबैक करें

6. **लॉगिंग और निगरानी**
   - फ़ंक्शन निष्पादन समय ट्रैक करें
   - पैरामीटर और परिणाम लॉग करें
   - उपयोग पैटर्न की निगरानी करें

### 🔑 डेकोरेटर्स से प्रमुख अंतर:

| फीचर | डेकोरेटर | मिडलवेयर |
|---------|-----------|------------|
| **स्कोप** | एकल फ़ंक्शन | एजेंट में सभी फ़ंक्शन |
| **लचीलापन** | परिभाषा पर निश्चित | रनटाइम पर गतिशील |
| **संदर्भ** | सीमित | पूर्ण एजेंट संदर्भ |
| **संयोजन** | कई डेकोरेटर | मिडलवेयर पाइपलाइन |
| **एजेंट-जागरूक** | नहीं | हाँ (एजेंट स्टेट तक पहुंच) |

### 📚 कब मिडलवेयर का उपयोग करें:

✅ **मिडलवेयर का उपयोग करें जब:**
- आपको उपयोगकर्ता/सत्र स्थिति के आधार पर व्यवहार संशोधित करना हो
- आप कई फ़ंक्शन्स पर तर्क लागू करना चाहते हों
- आपको एजेंट-स्तरीय संदर्भ की आवश्यकता हो
- आप क्रॉस-कटिंग चिंताओं (लॉगिंग, प्रमाणीकरण, आदि) को लागू कर रहे हों

❌ **मिडलवेयर का उपयोग न करें जब:**
- सरल इनपुट सत्यापन (पायडेंटिक का उपयोग करें)
- फ़ंक्शन-विशिष्ट तर्क (फ़ंक्शन में ही रखें)
- एक बार संशोधन (सिर्फ फ़ंक्शन बदलें)

### 🎓 उन्नत पैटर्न:

```python
# कई मिडलवेयर (कार्य निष्पादन क्रम महत्वपूर्ण है!)
middleware=[
    logging_middleware,      # पहले लॉग्स
    auth_middleware,         # फिर प्रमाणीकरण जांचता है
    cache_middleware,        # फिर कैश जांचता है
    rate_limit_middleware,   # फिर दर सीमाएं लागू करता है
    priority_check_middleware  # अंत में प्राथमिकता जांच
]

# सशर्त मिडलवेयर निष्पादन
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # परिणाम संशोधित करें
    else:
        # पूरी तरह से निष्पादन छोड़ें
        context.result = cached_value
```

### 🔗 संबंधित अवधारणाएँ:

- **एजेंट मिडलवेयर**: एजेंट.run() कॉल्स को इंटरसेप्ट करता है
- **फ़ंक्शन मिडलवेयर**: टूल फ़ंक्शन कॉल्स को इंटरसेप्ट करता है (जैसा कि हमने उपयोग किया!)
- **मिडलवेयर पाइपलाइन**: मिडलवेयर की एक श्रृंखला जो क्रमशः निष्पादित होती है
- **संदर्भ प्रचारण**: मिडलवेयर श्रृंखला से माध्यम से स्थिति पास करना


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
